# Shape Flow Demo

This notebook gives a minimal end-to-end test of the pipeline:

1. Generate synthetic `e_true`, `e_meas`, and `cond` arrays.
2. Train a conditional Zuko NSF likelihood model.
3. Evaluate standardized and physical-unit log likelihoods.
4. Save, reload, and compare likelihood values.
5. Draw `emcee` posterior samples for `e_true` from the learned likelihood.

From the repository root, install the demo dependencies with:

```bash
python -m pip install -e ".[demo]"
```

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

ROOT = Path.cwd()
if not (ROOT / "shape_flow").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shape_flow import (
    MCMCConfig,
    TrainingConfig,
    load_likelihood,
    sample_posterior_emcee,
    train_shape_flow,
)

print(f"repo root: {ROOT}")
print(f"torch: {torch.__version__}")
print(f"device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## Generate Synthetic Data

The measured shape is a noisy, weakly biased version of the intrinsic shape.
The noise level depends on the condition variables, especially signal-to-noise
and half-light radius.

In [ ]:
rng = np.random.default_rng(123)
torch.manual_seed(123)

n_samples = 1500

e_true = rng.normal(0.0, 0.22, size=(n_samples, 2)).astype("float32")
e_true = np.clip(e_true, -0.8, 0.8)

snr = rng.uniform(15.0, 100.0, size=n_samples)
half_light_radius = rng.uniform(0.4, 2.2, size=n_samples)
magnitude = rng.uniform(20.0, 25.0, size=n_samples)
cond = np.column_stack([snr, half_light_radius, magnitude]).astype("float32")

noise_scale = 0.015 + 0.20 / snr + 0.025 / half_light_radius
bias = np.column_stack(
    [
        0.015 * e_true[:, 0] + 0.002 * (magnitude - 22.5),
        -0.010 * e_true[:, 1] + 0.001 * (half_light_radius - 1.3),
    ]
).astype("float32")
noise = rng.normal(0.0, noise_scale[:, None], size=(n_samples, 2)).astype("float32")
e_meas = (e_true + bias + noise).astype("float32")

print(f"e_true: {e_true.shape}")
print(f"e_meas: {e_meas.shape}")
print(f"cond: {cond.shape}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(e_true[:400, 0], e_meas[:400, 0], s=8, alpha=0.45)
axes[0].plot([-0.8, 0.8], [-0.8, 0.8], color="black", linewidth=1)
axes[0].set_xlabel("true e1")
axes[0].set_ylabel("measured e1")
axes[0].set_title("Synthetic shape measurements")

axes[1].scatter(snr[:400], np.abs(e_meas[:400, 0] - e_true[:400, 0]), s=8, alpha=0.45)
axes[1].set_xlabel("signal-to-noise")
axes[1].set_ylabel("absolute e1 error")
axes[1].set_title("Heteroscedastic measurement error")

fig.tight_layout()

## Train A Small Flow

This demo uses a deliberately small model and few epochs so it runs quickly.
For real calibration work, increase `epochs`, `hidden_features`, and
`transforms`, then monitor validation NLL.

In [ ]:
checkpoint = ROOT / "artifacts" / "demo_shape_flow.pt"

config = TrainingConfig(
    epochs=8,
    batch_size=256,
    val_fraction=0.2,
    learning_rate=2.0e-3,
    weight_decay=1.0e-4,
    grad_clip_norm=5.0,
    transforms=2,
    hidden_features=(32, 32),
    bins=8,
    seed=123,
    device="cuda" if torch.cuda.is_available() else "cpu",
    checkpoint_path=checkpoint,
)


def progress(epoch: int, metrics: dict[str, float]) -> None:
    print(
        f"epoch={epoch:03d} "
        f"train_nll={metrics['train_nll']:.4f} "
        f"val_nll={metrics['val_nll']:.4f}"
    )


result = train_shape_flow(
    e_true,
    e_meas,
    cond,
    config=config,
    progress_callback=progress,
)

print(f"best epoch: {result.best_epoch}")
print(f"best validation NLL: {result.best_val_nll:.4f}")
print(f"saved checkpoint: {checkpoint}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(result.history["train_nll"], label="train")
ax.plot(result.history["val_nll"], label="validation")
ax.set_xlabel("epoch index")
ax.set_ylabel("negative log likelihood")
ax.legend()
ax.set_title("Training curve")
fig.tight_layout()

## Evaluate Likelihoods

`log_prob_standardized` is the learned density in standardized target units.
`log_prob` subtracts the target-scaler log-Jacobian and returns the
physical-unit likelihood.

In [ ]:
idx = np.arange(8)
log_q_std = result.likelihood.log_prob_standardized(e_meas[idx], e_true[idx], cond[idx])
log_q = result.likelihood.log_prob(e_meas[idx], e_true[idx], cond[idx])

print("standardized log q:")
print(log_q_std.numpy())
print("physical-unit log q:")
print(log_q.numpy())
print("target log-Jacobian:")
print(result.likelihood.target_scaler.log_abs_det_jacobian().cpu().item())

## Reload The Checkpoint

This checks that the model and both scalers are saved and restored together.

In [ ]:
reloaded = load_likelihood(checkpoint, map_location="cpu")
log_q_reloaded = reloaded.log_prob(e_meas[idx], e_true[idx], cond[idx])

torch.testing.assert_close(log_q.cpu(), log_q_reloaded.cpu(), rtol=1.0e-5, atol=1.0e-5)
print("reload check passed")

## Sample The Intrinsic-Shape Posterior

For a fixed measured shape and condition vector, `emcee` samples the Bayesian
inverse problem `p(e_true | e_meas, cond)`. The default prior is uniform inside
the ellipticity disk `sqrt(e1**2 + e2**2) <= prior_radius`.

In [ ]:
sample_id = 0
mcmc_config = MCMCConfig(
    n_walkers=32,
    n_steps=300,
    burn_in=100,
    thin=2,
    initial_scale=0.03,
    prior_radius=0.999,
    random_seed=321,
)

posterior = sample_posterior_emcee(
    reloaded,
    e_meas[sample_id],
    cond[sample_id],
    config=mcmc_config,
)

posterior_path = ROOT / "artifacts" / "demo_posterior_samples.npz"
np.savez(
    posterior_path,
    samples=posterior.samples,
    log_prob=posterior.log_prob,
    chain=posterior.chain,
    acceptance_fraction=posterior.acceptance_fraction,
    e_meas=e_meas[sample_id],
    cond=cond[sample_id],
)

print(f"posterior samples: {posterior.samples.shape}")
print(f"mean acceptance: {posterior.acceptance_fraction.mean():.3f}")
print(f"posterior mean e_true: {posterior.samples.mean(axis=0)}")
print(f"actual synthetic e_true: {e_true[sample_id]}")
print(f"saved posterior samples: {posterior_path}")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(posterior.samples[:, 0], posterior.samples[:, 1], s=5, alpha=0.15)
ax.scatter(*e_true[sample_id], color="black", marker="x", s=70, label="true")
ax.scatter(*e_meas[sample_id], color="tab:red", marker="+", s=80, label="measured")
ax.set_xlabel("intrinsic e1")
ax.set_ylabel("intrinsic e2")
ax.set_title("MCMC posterior samples")
ax.legend()
fig.tight_layout()

## Simple Sanity Checks

The observed measured shapes should usually score better than deliberately
shifted measured shapes under the same context.

In [ ]:
check = np.arange(128)
shifted_e_meas = e_meas[check].copy()
shifted_e_meas[:, 0] += 0.25

log_q_observed = reloaded.log_prob(e_meas[check], e_true[check], cond[check])
log_q_shifted = reloaded.log_prob(shifted_e_meas, e_true[check], cond[check])

print(f"mean log q observed: {log_q_observed.mean().item():.3f}")
print(f"mean log q shifted:  {log_q_shifted.mean().item():.3f}")

In [ ]:
sample_id = 0
grid = np.linspace(e_meas[sample_id, 0] - 0.25, e_meas[sample_id, 0] + 0.25, 160)
candidate_e_meas = np.repeat(e_meas[sample_id : sample_id + 1], len(grid), axis=0)
candidate_e_meas[:, 0] = grid.astype("float32")
candidate_e_true = np.repeat(e_true[sample_id : sample_id + 1], len(grid), axis=0)
candidate_cond = np.repeat(cond[sample_id : sample_id + 1], len(grid), axis=0)

log_q_curve = reloaded.log_prob(candidate_e_meas, candidate_e_true, candidate_cond).numpy()

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(grid, log_q_curve)
ax.axvline(e_true[sample_id, 0], color="black", linewidth=1, label="true e1")
ax.axvline(e_meas[sample_id, 0], color="tab:red", linewidth=1, label="measured e1")
ax.set_xlabel("candidate measured e1")
ax.set_ylabel("log q")
ax.set_title("One-dimensional likelihood slice")
ax.legend()
fig.tight_layout()